In [ ]:
!pip install openjij pyqubo

In [ ]:
import numpy as np
import pandas as pd
df=pd.read_csv('/content/survey lung cancer.csv')

In [ ]:
dfen2 = df.copy()
dfen2['GENDER'] = dfen2['GENDER'].map({'F':0,'M':1})
dfen2['LUNG_CANCER'] = dfen2['LUNG_CANCER'].map({'NO':0,'YES':1})

In [ ]:
# ------------------------------------------
#  Install imblearn once (if not installed)
# ------------------------------------------
# !pip install imbalanced-learn

from imblearn.over_sampling import SMOTE
from collections import Counter
import pandas as pd

# ------------------------------------------
# 1️⃣  Verify dataset & identify target column
# ------------------------------------------
# You already have dfs
print("Original shape:", dfen2.shape)
print("Original class distribution:")
print(dfen2['LUNG_CANCER'].value_counts())

# ------------------------------------------
# 2️⃣  Separate features (X) and target (y)
# ------------------------------------------
X = dfen2.drop(columns=['LUNG_CANCER']).values
y = dfen2['LUNG_CANCER'].values

# ------------------------------------------
# 3️⃣  Apply SMOTE only on the minority class
# ------------------------------------------
smote = SMOTE(random_state=42)
X_res, y_res = smote.fit_resample(X, y)

# ------------------------------------------
# 4️⃣  Rebuild into a balanced DataFrame
# ------------------------------------------
columns = dfen2.drop(columns=['LUNG_CANCER']).columns
dfs_smotes = pd.DataFrame(X_res, columns=columns)
dfs_smotes['LUNG_CANCER'] = y_res


# ------------------------------------------
# 5️⃣  Check the new class balance
# ------------------------------------------
print("\nAfter SMOTE:")
print(dfs_smotes['LUNG_CANCER'].value_counts())
print("New shape:", dfs_smotes.shape)

In [ ]:
dff=dfs_smotes[:]
dff

In [ ]:
dff.info()

**<h1>Asvm**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)

import openjij as oj
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
NUM_REPEATS = 65
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "openjij_annealing_svm.csv"

TARGET = "LUNG_CANCER"
FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

# ============================================================
# SYNTHETIC DATA (DEMO)
# ============================================================
try:
    dfs_smotes
except NameError:
    np.random.seed(42)
    n_samples = 200
    n_features = 15

    X_pos = np.random.randn(n_samples//2,n_features)+1
    X_neg = np.random.randn(n_samples//2,n_features)-1

    X = np.vstack([X_pos,X_neg])
    y = np.array([1]*(n_samples//2)+[0]*(n_samples//2))

    cols = [f"feature_{i}" for i in range(n_features)]

    dfs_smotes = pd.DataFrame(X,columns=cols)
    dfs_smotes[TARGET] = y

X_raw = dfs_smotes.iloc[:,FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print("Using features:",feature_names)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true,y_prob,thr=0.5):

    y_pred=(y_prob>=thr).astype(int)

    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()

    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC-AUC":roc_auc_score(y_true,y_prob),
        "PR-AUC":average_precision_score(y_true,y_prob),
        "Sensitivity":tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity":tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa":cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# LINEAR KERNEL
# ============================================================
def compute_linear_kernel(X1,X2):
    return X1 @ X2.T

# ============================================================
# QUBO SOLVER (OPENJIJ)
# ============================================================
def solve_dual_openjij(K,y,C=1.0,num_reads=50,num_sweeps=500):

    n=len(y)
    n_bits=4

    scale=C/(2**n_bits-1)

    Q={}

    for i in range(n):
        for ki in range(n_bits):

            idx_i=i*n_bits+ki
            coeff_i=scale*(2**ki)

            Q[(idx_i,idx_i)]=Q.get((idx_i,idx_i),0)-coeff_i

            for j in range(n):
                for kj in range(n_bits):

                    idx_j=j*n_bits+kj
                    coeff_j=scale*(2**kj)

                    key=(min(idx_i,idx_j),max(idx_i,idx_j))

                    Q[key]=Q.get(key,0)+0.5*y[i]*y[j]*K[i,j]*coeff_i*coeff_j

    # equality constraint
    penalty=10

    for i in range(n):
        for ki in range(n_bits):

            idx_i=i*n_bits+ki
            coeff_i=scale*(2**ki)*y[i]

            Q[(idx_i,idx_i)]+=penalty*coeff_i**2

            for j in range(i+1,n):
                for kj in range(n_bits):

                    idx_j=j*n_bits+kj
                    coeff_j=scale*(2**kj)*y[j]

                    key=(min(idx_i,idx_j),max(idx_i,idx_j))

                    Q[key]=Q.get(key,0)+2*penalty*coeff_i*coeff_j

    sampler=oj.SQASampler()

    response=sampler.sample_qubo(
        Q,
        num_reads=num_reads,
        num_sweeps=num_sweeps
    )

    sample=response.first.sample

    alphas=np.zeros(n)

    for i in range(n):

        val=sum((2**k)*sample.get(i*n_bits+k,0) for k in range(n_bits))

        alphas[i]=scale*val

    return alphas

# ============================================================
# PREDICTION
# ============================================================
def predict_openjij_svm(X_train,y_train,X_test,C=1):

    K_train=compute_linear_kernel(X_train,X_train)
    K_test=compute_linear_kernel(X_test,X_train)

    y_svm=np.where(y_train==0,-1,1)

    alphas=solve_dual_openjij(K_train,y_svm,C=C)

    decision = K_test @ (alphas*y_svm)

    return 1/(1+np.exp(-decision))

# ============================================================
# MAIN LOOP
# ============================================================
summary=[]

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test = train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    y_prob=predict_openjij_svm(X_train,y_train,X_test)

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    print(
        #f"[{progress:6.2f}%] "
        f"QSVM_Fixed | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row={
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)

# ============================================================
# FINAL
# ============================================================
summary_df=pd.DataFrame(summary).sort_values(
    "Accuracy",
    ascending=False
)

print("\nFINAL RESULTS")
print(summary_df)

**<h1>annealing knn**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)
from sklearn.metrics.pairwise import linear_kernel

import openjij as oj
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
NUM_REPEATS = 65
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "openjij_annealing_knn.csv"

TARGET = "LUNG_CANCER"
FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]
K_NEIGHBORS = 3        # number of neighbors
NUM_READS = 50         # annealing runs

# ============================================================
# SYNTHETIC DATA (DEMO)
# ============================================================
try:
    dfs_smotes
except NameError:
    np.random.seed(42)
    n_samples = 200
    n_features = 15

    X_pos = np.random.randn(n_samples//2,n_features)+1
    X_neg = np.random.randn(n_samples//2,n_features)-1

    X = np.vstack([X_pos,X_neg])
    y = np.array([1]*(n_samples//2)+[0]*(n_samples//2))

    cols = [f"feature_{i}" for i in range(n_features)]
    dfs_smotes = pd.DataFrame(X,columns=cols)
    dfs_smotes[TARGET] = y

X_raw = dfs_smotes.iloc[:,FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)
feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print("Using features:",feature_names)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true,y_pred):
    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()
    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC-AUC":roc_auc_score(y_true,y_pred),
        "PR-AUC":average_precision_score(y_true,y_pred),
        "Sensitivity":tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity":tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa":cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# LINEAR KERNEL
# ============================================================
def compute_linear_kernel(X1,X2):
    return X1 @ X2.T

# ============================================================
# ANNEALING KNN
# ============================================================
def annealing_knn_predict(X_train, y_train, X_test, k=3, num_reads=50, lam=5):
    """
    Annealing-based k-Nearest Neighbors using OpenJij
    """
    sampler = oj.SASampler()
    y_pred = []

    for x in X_test:
        # compute similarity to all training points
        sim = linear_kernel(x.reshape(1,-1), X_train).flatten()

        # build QUBO to select k neighbors maximizing similarity
        n = len(sim)
        Q = {}
        for i in range(n):
            Q[(i,i)] = -sim[i] + lam*(1 - 2*k)
            for j in range(i+1, n):
                Q[(i,j)] = 2*lam

        # solve QUBO
        response = sampler.sample_qubo(Q, num_reads=num_reads)
        sample = response.first.sample

        neighbors = [i for i,v in sample.items() if v==1]
        if len(neighbors)==0:
            y_pred.append(0)
            continue

        labels = y_train[neighbors]
        pred = 1 if np.sum(labels) >= len(labels)/2 else 0
        y_pred.append(pred)

    return np.array(y_pred)

# ============================================================
# MAIN LOOP
# ============================================================
summary = []

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1, NUM_REPEATS+1):

    X_train,X_test,y_train,y_test = train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp = SimpleImputer(strategy="median")
    sc = StandardScaler()
    X_train = sc.fit_transform(imp.fit_transform(X_train))
    X_test = sc.transform(imp.transform(X_test))

    start = time.time()
    y_pred = annealing_knn_predict(X_train, y_train, X_test, k=K_NEIGHBORS, num_reads=NUM_READS)
    runtime = time.time()-start

    metrics = compute_metrics(y_test, y_pred)

    print(
        f"Annealing_KNN | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row = {
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)
    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)

# ============================================================
# FINAL
# ============================================================
summary_df = pd.DataFrame(summary).sort_values("Accuracy", ascending=False)
print("\nFINAL RESULTS")
print(summary_df)

**<h1>AQboost**

In [ ]:
# ============================================================
# IMPORTS
# ============================================================
import numpy as np
import pandas as pd
import os
import time

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score,
    f1_score, roc_auc_score, average_precision_score,
    confusion_matrix, cohen_kappa_score
)

from sklearn.tree import DecisionTreeClassifier

import openjij as oj
import warnings
warnings.filterwarnings("ignore")

# ============================================================
# USER CONTROLS
# ============================================================
NUM_REPEATS = 65
TEST_SIZE = 0.8
RANDOM_SEED_BASE = 42
AUTO_SAVE_PATH = "openjij_qboost.csv"

TARGET = "LUNG_CANCER"
FIXED_FEATURES_IDX = [8,10,9,13,11,14,5,3,6,2]

N_WEAK = 20
LAMBDA = 0.1

# ============================================================
# SYNTHETIC DATA (DEMO)
# ============================================================
try:
    dfs_smotes
except NameError:
    print("Generating synthetic data...")
    np.random.seed(42)
    n_samples = 200
    n_features = 15

    X_pos = np.random.randn(n_samples//2,n_features)+1
    X_neg = np.random.randn(n_samples//2,n_features)-1

    X = np.vstack([X_pos,X_neg])
    y = np.array([1]*(n_samples//2)+[0]*(n_samples//2))

    cols = [f"feature_{i}" for i in range(n_features)]

    dfs_smotes = pd.DataFrame(X,columns=cols)
    dfs_smotes[TARGET] = y

X_raw = dfs_smotes.iloc[:,FIXED_FEATURES_IDX].values.astype(float)
y = dfs_smotes[TARGET].values.astype(int)

feature_names = dfs_smotes.columns[FIXED_FEATURES_IDX].tolist()
print("Using features:",feature_names)

# ============================================================
# METRICS
# ============================================================
def compute_metrics(y_true,y_prob,thr=0.5):

    y_pred=(y_prob>=thr).astype(int)

    tn,fp,fn,tp = confusion_matrix(y_true,y_pred).ravel()

    return {
        "Accuracy":accuracy_score(y_true,y_pred),
        "Precision":precision_score(y_true,y_pred,zero_division=0),
        "Recall":recall_score(y_true,y_pred,zero_division=0),
        "F1":f1_score(y_true,y_pred,zero_division=0),
        "ROC-AUC":roc_auc_score(y_true,y_prob),
        "PR-AUC":average_precision_score(y_true,y_prob),
        "Sensitivity":tp/(tp+fn) if(tp+fn)>0 else 0,
        "Specificity":tn/(tn+fp) if(tn+fp)>0 else 0,
        "Kappa":cohen_kappa_score(y_true,y_pred)
    }

# ============================================================
# GENERATE WEAK CLASSIFIERS
# ============================================================
def build_weak_classifiers(X_train,y_train,n_weak=N_WEAK):

    models=[]
    preds=[]

    for _ in range(n_weak):

        stump = DecisionTreeClassifier(max_depth=1)

        idx = np.random.choice(len(X_train),len(X_train),replace=True)

        stump.fit(X_train[idx],y_train[idx])

        pred = stump.predict(X_train)

        pred = np.where(pred==0,-1,1)

        models.append(stump)
        preds.append(pred)

    return models,np.array(preds)

# ============================================================
# QBOOST SOLVER
# ============================================================
def solve_qboost(preds,y_train):

    y_spin = np.where(y_train==0,-1,1)

    M = preds.shape[0]

    Q={}

    for i in range(M):
        for j in range(M):

            val = np.sum(preds[i]*preds[j]) / len(y_train)

            key=(min(i,j),max(i,j))

            Q[key] = Q.get(key,0) + val

    for i in range(M):

        val = -2*np.sum(y_spin*preds[i]) / len(y_train)

        Q[(i,i)] = Q.get((i,i),0) + val + LAMBDA

    sampler = oj.SQASampler()

    response = sampler.sample_qubo(
        Q,
        num_reads=200,
        num_sweeps=2000
    )

    sample=response.first.sample

    w = np.array([sample.get(i,0) for i in range(M)])

    return w

# ============================================================
# PREDICTION
# ============================================================
def predict_qboost(X_train,y_train,X_test):

    models,preds = build_weak_classifiers(X_train,y_train)

    weights = solve_qboost(preds,y_train)

    agg_train = np.zeros(len(X_train))

    for w,m in zip(weights,models):
        if w==1:
            p = m.predict(X_train)
            p = np.where(p==0,-1,1)
            agg_train += p

    bias = -np.mean(agg_train)

    agg_test = np.zeros(len(X_test))

    for w,m in zip(weights,models):
        if w==1:
            p = m.predict(X_test)
            p = np.where(p==0,-1,1)
            agg_test += p

    decision = agg_test + bias

    return 1/(1+np.exp(-decision))

# ============================================================
# MAIN LOOP
# ============================================================
summary=[]

if os.path.exists(AUTO_SAVE_PATH):
    os.remove(AUTO_SAVE_PATH)

for split in range(1,NUM_REPEATS+1):

    X_train,X_test,y_train,y_test = train_test_split(
        X_raw,y,
        test_size=TEST_SIZE,
        stratify=y,
        random_state=RANDOM_SEED_BASE*split
    )

    imp=SimpleImputer(strategy="median")
    sc=StandardScaler()

    X_train=sc.fit_transform(imp.fit_transform(X_train))
    X_test=sc.transform(imp.transform(X_test))

    start=time.time()

    y_prob=predict_qboost(X_train,y_train,X_test)

    runtime=time.time()-start

    metrics=compute_metrics(y_test,y_prob)

    print(
        f"QBOOST | "
        f"Acc={metrics['Accuracy']:.4f} | "
        f"ROC-AUC={metrics['ROC-AUC']:.4f} | "
        f"F1={metrics['F1']:.4f} | "
        f"Precision={metrics['Precision']:.4f} | "
        f"Sensitivity={metrics['Sensitivity']:.4f} | "
        f"Specificity={metrics['Specificity']:.4f} | "
        f"Kappa={metrics['Kappa']:.4f} | "
        f"Time={runtime:.2f}s"
    )

    row={
        "Split":split,
        "Accuracy":metrics["Accuracy"],
        "ROC_AUC":metrics["ROC-AUC"],
        "F1":metrics["F1"],
        "Precision":metrics["Precision"],
        "Sensitivity":metrics["Sensitivity"],
        "Specificity":metrics["Specificity"],
        "Kappa":metrics["Kappa"],
        "Runtime_sec":runtime
    }

    summary.append(row)

    pd.DataFrame(summary).to_csv(AUTO_SAVE_PATH,index=False)

# ============================================================
# FINAL
# ============================================================
summary_df=pd.DataFrame(summary).sort_values(
    "Accuracy",
    ascending=False
)

print("\nFINAL RESULTS")
print(summary_df)